有数据时，先明确每行每列以及单位  
检查缺失值  
异常值（离谱的，需判断真实极端还是录入错误）  
重复值（记录多次）  
数据类型错误（数字类型变成字符串有问题）  
把不同单位，不同方向，不同尺度数据转换为可比较形式  
通过统计量、图表、相关性分析，为模型选择提供依据  
评委不仅看结果，还看数据来源、处理依据和是否可复现  
目标：让数据被模型正确使用，将附件整理为可直接使用的数值矩阵  

X=[x_ij]_n*m  
n个评价对象，m个指标，x_ij表示第i个对象在第j个指标上的观测值  

对象——指标——数据矩阵——模型输入——结果输出  

注意：  
缺失值一定要处理，指标方向要统一（越大/小越好）  
量纲要消除  

每一行称样本，每一列称指标，值为观测结果  

常见数据格式：  
Excel：`.xlsx` → `pd.read_excel(路径)`  
CSV：`.csv` → `pd.read_csv(路径)`  
图片文本先提取信息，一般不需要导入  

读取数据检查顺序：  
文件是否存在  
编码是否正确（是否乱码，重下）  
列名是否完整  
数据类型是否合理  
是否可保存  

In [1]:
# -*- coding: utf-8 -*-
# 环境准备 + 全 Notebook 唯一的配置区（换数据只改 CFG）
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


def find_root(marker="导入数据"):
    """向上找含 导入数据/ 的项目根：换电脑、换目录都不用改路径。"""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).is_dir():
            return p
    return None


ROOT = find_root() or Path(r"d:\git项目")
DATA_DIR = ROOT / "导入数据"
OUT_DIR = ROOT / "输出数据"

# ↓↓↓ 换题目只改这一个字典 ↓↓↓
CFG = dict(
    raw_stem="mcm_final_raw",                    # 原始数据文件名（不含扩展名）
    sample_stem="初始资金为10000的资金变化",      # 示例数据
    clean_stem="final_cleaned",                  # 清洗结果
    out_3sigma="result_3sigma",
    out_iqr="result_iqr",
    sheet="最终结果",
    num_cols=["pm25", "temperature", "traffic_index", "survey_score", "household_income"],
    cat_cols=["date", "region", "fuel_type"],
    key_cols=["date", "region"],                 # 去重主键
    group_col="region",                          # 分组统计 / 画图的分组列
    group_label="区域",                          # 图标题里的分组中文名
    plot_prefix="boxplot",                       # 箱线图文件名前缀
    detect_cols=["pm25", "temperature", "traffic_index", "survey_score"],
    na_markers=["", "NA", "缺测", "未填", "999", 999, "9999", 9999],
    drop_missing_rate=50.0,
    fill_median=["traffic_index"],
    fill_mode=["fuel_type"],
    fill_group_median=["pm25"],
    fill_interp=["temperature"],
    keep_na=["survey_score"],                    # 有信息缺失：保留 NaN + 指示列
    sort_cols=["region", "date"],
)


def load_any(folder, stem, order=(".csv", ".xlsx")):
    """优先 csv（utf-8-sig 去 BOM），其次 xlsx；找不到给出完整路径。"""
    for ext in order:
        p = folder / f"{stem}{ext}"
        if p.exists():
            print("读取：", p)
            return (pd.read_excel(p) if ext == ".xlsx"
                    else pd.read_csv(p, encoding="utf-8-sig"))
    raise FileNotFoundError(f"{folder} 下找不到 {stem}.csv / .xlsx")


def save_both(d, folder, stem, sheet):
    """CSV（带 BOM）+ Excel 一起写，输出目录不存在时自动建。"""
    folder.mkdir(parents=True, exist_ok=True)
    d.to_csv(folder / f"{stem}.csv", index=False, encoding="utf-8-sig")
    d.to_excel(folder / f"{stem}.xlsx", index=False, sheet_name=sheet)
    print("已输出：", folder / f"{stem}.xlsx")


print("项目根目录：", ROOT)
print("数值列：", CFG["num_cols"])

# -*- coding: utf-8 -*-
# 一行换数据：自动读文件、分列、推断缺失填补与输出名，再打补丁到 CFG
# 用法见下方 use_data 的文档字符串；本函数只改 CFG，不读写输出数据
def use_data(raw_stem, **overrides):
    """换题目只需这一行：use_data("文件名")（不含扩展名，文件放在 导入数据/）。

    自动完成：识别数值列/类别列 → 把哨兵与缺失算成 NaN → 推断去重主键、
    分组列、异常检测列、缺失填补列、输出文件名。需要人工判断的用关键字覆盖：
        use_data("lake_water_2024", key_cols=["date","station"],
                 keep_na=["complaint_score"], group_col="station")
    覆盖项可以是 CFG 里的任意字段；本函数只改写出来的键，其余保持默认。
    """
    raw = load_any(DATA_DIR, raw_stem)
    raw.columns = [str(c).strip() for c in raw.columns]
    chk = raw.replace(CFG["na_markers"], np.nan)      # 哨兵值先当成缺失

    # ① 列类型：能转成数字的比例 > 90% 视为数值列（带空格的文本数字也算）
    num, cat = [], []
    for c in raw.columns:
        s = chk[c]
        ok = pd.to_numeric(s.astype(str).str.strip(), errors="coerce").notna().sum() / max(s.notna().sum(), 1)
        (num if ok > .9 else cat).append(c)

    # ② 高缺失列会被清洗单元删掉，不参与检测与填补
    keep_cols = [c for c in chk.columns if chk[c].isna().mean() * 100 < CFG["drop_missing_rate"]]
    fill_num = [c for c in num if c in keep_cols and chk[c].isna().any()]
    fill_cat = [c for c in cat if c in keep_cols and chk[c].isna().any()]

    # ③ 猜日期列、分组列（类别数 2~20）、去重主键与插值排序
    dates = [c for c in cat if any(k in c.lower() for k in
             ("date", "time", "日期", "时间", "月份", "year", "年", "月"))]
    groups = [c for c in cat if 1 < chk[c].nunique(dropna=True) <= 20 and c not in dates]
    g = groups[0] if groups else (cat[0] if cat else "")
    key = (dates[:1] + ([g] if g and g not in dates else [])) or cat[:2]
    sort = ([g] + dates[:1]) if (g and dates) else (dates[:1] or cat[:1])

    auto = dict(
        raw_stem=raw_stem, num_cols=num, cat_cols=cat,
        key_cols=key, group_col=g, group_label=g,
        detect_cols=[c for c in num if c in keep_cols],
        fill_median=fill_num, fill_mode=fill_cat,
        fill_group_median=[], fill_interp=[], keep_na=[], sort_cols=sort,
        clean_stem=f"{raw_stem}_clean", out_3sigma=f"{raw_stem}_3sigma",
        out_iqr=f"{raw_stem}_iqr", plot_prefix=f"{raw_stem}_box",
    )
    auto.update(overrides)
    for c in auto["keep_na"]:                       # 保留 NaN 的列不参与填补
        for f in ("fill_median", "fill_mode", "fill_group_median", "fill_interp"):
            auto[f] = [x for x in auto[f] if x != c]
    CFG.update(auto)
    print(f"[换数据] {raw_stem}：数值列 {len(num)} 个，类别列 {len(cat)} 个")
    print("  数值列 =", num)
    print("  类别列 =", cat)
    print("  主键 =", key, "| 分组 =", g, "| 检测列 =", auto["detect_cols"])
    print("  填补 =", auto["fill_median"] + auto["fill_mode"], "| 保留NaN =", auto["keep_na"])
    print("  输出 =", auto["clean_stem"], "/", auto["out_3sigma"], "/", auto["out_iqr"])
    return CFG


d:\git项目\刘兴博的笔记
True
True


In [ ]:
# ===== 换题目只改下面这一行 =====
# 继续用原题做笔记 → 保持 None
# 换题 → 填新文件名（不含 .csv / .xlsx，文件放在 导入数据/ 里）
NEW_DATA = None            # 例如：NEW_DATA = "lake_water_2024"

if NEW_DATA:
    use_data(NEW_DATA)     # 自动识别列、推断配置，然后 Run All
    # 需要人工指定时写：use_data(NEW_DATA, keep_na=["某列"], group_col="地区")


In [2]:
# 示例：把附件读成 DataFrame（新项目里没有这个示例文件时自动跳过，不影响主流程）
try:
    df_raw = load_any(DATA_DIR, CFG["sample_stem"])
    df_raw
except FileNotFoundError:
    df_raw = None
    print(f"[提示] 示例数据 {CFG['sample_stem']} 不存在，已跳过；不影响后续清洗流程。")

,日期(月/日/年),资金
0,9/11/16,1.000000e+03
1,9/12/16,1.000000e+04
2,9/13/16,1.002050e+04
3,9/14/16,9.986060e+03
4,9/15/16,1.001165e+04
...,...,...
1821,9/6/21,3.798801e+10
1822,9/7/21,3.865454e+10
1823,9/8/21,3.789661e+10
1824,9/9/21,3.789661e+10


In [3]:
if df_raw is not None:
    print(df_raw.shape)   # 输出数据的行列数
    df_raw.info()         # 输出数据的基本信息
else:
    print("示例数据未加载，跳过检查（不影响主流程）")

(1826, 2)
<class 'pandas.DataFrame'>
RangeIndex: 1826 entries, 0 to 1825
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   日期(月/日/年)  1826 non-null   str    
 1   资金         1826 non-null   float64
dtypes: float64(1), str(1)
memory usage: 28.7 KB


数据清洗学习  
1、把错误不完整不一致和异常记录进行检查和处理  
查看字段——检查确实——检查重复——检查类型——识别异常——记录处理  
2、清洗原则：说清楚每一步原因，并能在这里复现（用代码）  
原始附件——处理代码——输出表格  
3、论文中说明：处理对象、处理方法、处理依据，以及处理前后变化  

缺失值常见来源：  
1、数据采集时没记录  
2、问卷中受访者未填写  
3、设备故障中断  
4、多个数据表合并字段不匹配  
5、原始数据中使用特殊符号表示缺测  
影响均值、方差、相关系数、回归模型、聚类模型和综合评价结果  
缺失值出现在某一类样本中时，随意删除可能造成样本偏差，处理缺失值关键在于保留数据含义并减少偏差  

缺失率=缺失值个数/样本总数*100%  
0、某一列缺失率高时可能直接删除列，不删除行  
1、缺失很少且不影响样本代表性时删除行，但一般连续时间序列不可随便删去，可能会造成时间跨度不一致  
2、缺失值少时可用均值、中位数、众数、分组中位数填补(分类明显)  
（差异明显时按区域分组，计算中位数，填补缺失再检查）尽量尊重数据分组特征  
3、时间序列或有顺序的数据可用插值（线性插值等）  
4、缺失本身有含义不能机械填补要单独说明  

df.duplicated()检查有无重复录入数据  
df.drop_duplicates()删除重复行  

In [9]:
# -*- coding: utf-8 -*-
# 数据清洗：规范化 → 去重 → 填补 → 插值 → 输出（列名/阈值全部来自 CFG）
# 单独运行本格前，请先运行第 1 个代码单元
def dedup(d, key, num):
    """删完全重复 + 主键冲突按确定性规则保留一条（有效字段多、偏离中位数小）。"""
    n = int(d.duplicated().sum())
    d = d.drop_duplicates(keep="first")
    conf = d.groupby(key)[num].nunique(dropna=False).gt(1).any(axis=1)
    med = d[num].median()
    d = d.assign(_ok=d[num].notna().sum(axis=1),
                 _dev=d[num].sub(med).abs().sum(axis=1))
    d = (d.sort_values(["_ok", "_dev"] + key,
                       ascending=[False, True] + [True] * len(key), kind="mergesort")
          .drop_duplicates(subset=key, keep="first")
          .drop(columns=["_ok", "_dev"]).reset_index(drop=True))
    print(f"完全重复 {n} 行已删 | 主键冲突组 {int(conf.sum())} 个 | 去重后 {d.shape}")
    return d


df = load_any(DATA_DIR, CFG["raw_stem"])
print("原始：", df.shape)

# ① 规范化：哨兵 -> NaN、数值列转数值、文本列去空格（自动跳过不存在的列）
df = df.replace(CFG["na_markers"], np.nan)
num = [c for c in CFG["num_cols"] if c in df.columns]
cat = [c for c in CFG["cat_cols"] if c in df.columns]
if num:
    df[num] = df[num].apply(pd.to_numeric, errors="coerce")
for c in cat:
    df[c] = df[c].astype("string").str.strip()

# ② 去重（必须先于填补，否则重复行会拉偏中位数）
key = [c for c in CFG["key_cols"] if c in df.columns]
df = dedup(df, key, num)
assert not df.duplicated().any() and not df.duplicated(subset=key).any()

# ③ 缺失处理：删列 -> 填补 -> 插值 -> 有信息缺失
rate = df.isna().mean() * 100
print("缺失率(%)：\n" + rate.round(2).to_string())
df = df.drop(columns=rate[rate >= CFG["drop_missing_rate"]].index)
for c in CFG["fill_median"]:
    if c in df:
        df[c] = df[c].fillna(df[c].median())
for c in CFG["fill_mode"]:
    if c in df:
        m = df[c].mode(dropna=True)
        df[c] = df[c].fillna(m.iat[0] if len(m) else "未知")
gcol = CFG["group_col"]
for c in CFG["fill_group_median"]:
    if c in df and gcol in df:
        g = df.groupby(gcol)[c].transform("median")
        df[c] = df[c].fillna(g).fillna(df[c].median())
sort_cols = [c for c in CFG["sort_cols"] if c in df.columns]
if sort_cols:
    df = df.sort_values(sort_cols)
for c in CFG["fill_interp"]:
    if c in df and gcol in df:
        df[c] = df.groupby(gcol)[c].transform(
            lambda s: s.interpolate(method="linear", limit_direction="both"))
for c in CFG["keep_na"]:
    if c in df:
        df[c + "_missing"] = df[c].isna().astype(int)

# ④ 复核：除 keep_na 外，参与建模的字段不应再有缺失
check = list(dict.fromkeys(
    [c for c in cat if c in df] +
    [c for c in (CFG["fill_median"] + CFG["fill_mode"] +
                 CFG["fill_group_median"] + CFG["fill_interp"]) if c in df]))
check = [c for c in check if c not in CFG["keep_na"]]
assert not df[check].isna().any().any(), f"仍有缺失：{df[check].isna().sum()}"
print("处理后缺失率(%)：\n" + (df.isna().mean() * 100).round(2).to_string())
save_both(df, OUT_DIR, CFG["clean_stem"], CFG["sheet"])

读取： d:\git项目\导入数据\mcm_final_raw.csv
原始： (381, 8)
完全重复 15 行已删 | 主键冲突组 6 个 | 去重后 (360, 8)
缺失率(%)：
date                 0.00
region               0.00
pm25                 4.44
temperature          1.11
traffic_index        2.50
fuel_type            2.22
survey_score         5.00
household_income    88.61


Data preprocessing. All non-standard missing markers (NA, 缺测, 未填, 9999, etc.) were converted to NaN, and text fields were stripped of leading/trailing spaces before de-duplication. We removed 15 exact duplicate rows and resolved 6 key conflicts on (date, region) using a deterministic rule that keeps the record with more valid fields and the smallest deviation from the overall median. The resulting panel contains 360 daily observations (4 regions × 90 consecutive days). Missing rate is defined as the number of missing cells divided by the total sample size, n = 360. The column household_income (88.61% missing) was dropped; columns, not rows, were removed so that the sample size was preserved. De-duplication was performed before imputation, since duplicate records would otherwise bias the group medians used for filling; interpolation was also placed after de-duplication to keep the time axis strictly 1-day spaced.  

Informative missingness in survey_score. The 18 missing values (5.00%) in survey_score arise from questionnaire non-response and are not missing at random: respondents with lower scores are less likely to report one. We therefore neither deleted these rows nor imputed them with a mean or median. Deletion would systematically remove low-score respondents and bias the mean, regression coefficients and composite scores upward, while imputation would erase the information carried by non-response itself. Instead, the NaNs were retained and a binary indicator survey_score_missing (1 = non-response) was added, allowing the response behaviour to enter the model explicitly. All 360 observations remain available for analyses that do not use survey_score.  

Why rows were not deleted. Row deletion is additionally inappropriate for a daily panel: removing the 18 incomplete rows breaks the strictly 1-day spacing (2-day gaps appear) and shifts regional sample shares by up to 0.73 percentage points, which would distort interpolation, time-series regression and clustering. For the models that require survey_score as an input, we use complete cases within that step only (n = 342) and conduct a sensitivity analysis under two extreme assumptions: (i) non-respondents take the 25th percentile of observed scores, and (ii) they take the regional mean. [fill in: state whether the ranking / coefficients / conclusions remain unchanged] This indicates [fill in: degree of robustness].  

⚠️ 最后那两句必须换成你实际跑出来的结果，不能照抄"结论稳健"——这是评审最容易追查的地方，也是学术诚信的底线。  

处理摘要表（评审扫一眼就懂，建议放正文或 Appendix）  

Variable | Missing rate | Mechanism / cause | Treatment | Rationale
--- | --- | --- | --- | ---
`household_income` | 88.61% | Almost uncollected | Drop column | Preserves n = 360
`pm25` | 4.44% | Missing markers; not recorded | Region-wise median | Respects regional heterogeneity
`temperature` | 1.11% | Equipment outage (consecutive) | Linear interpolation | Keeps 1-day spacing
`traffic_index` | 2.50% | Not recorded | Median | Robust to outliers
`fuel_type` | 2.22% | Blank questionnaire item | Mode | Categorical variable
`survey_score` | 5.00% | Non-response (MNAR) | Keep NaN + binary indicator | Missingness is informative


Assumptions & Limitations 里补一条：We treat non-response in survey_score as informative rather than ignorable; all conclusions involving this variable are conditional on the sensitivity analysis reported above.  

中文对照这几段的逻辑：第 1 段讲顺序（规范化 → 去重 → 填补，理由是重复行会拉偏分组中位数、重复日期会打乱插值）；第 2 段讲缺失机制是 MNAR，说明"不删也不填"是主动决策，不是遗漏；第 3 段用实测证据支撑 —— 我这边跑出来删行后间隔从 1 种变成 2 种（最大 2 天）、区域占比 ±0.73 个百分点、pm25 均值 51.51→51.59，这些数字可以直接引用，但记得在你自己的最终数据上复核一遍。  

翻译  
所有非标准缺失标记（NA、缺测、未填、9999 等）统一转成 NaN，去重前先去掉文本字段首尾空格。删掉 15 条完全重复行，并按确定性规则处理了 6 组 (date, region) 主键冲突：保留有效字段更多、且与整体中位数偏离最小的那条。最终是 360 条日度数据（4 个区域 × 90 个连续日）。缺失率 = 缺失值个数 ÷ 样本总数 ×100%（n=360）。household_income 缺失 88.61%，直接删列——只删列不删行，样本量不变。去重放在填补之前，因为重复记录会拉偏用于填补的分组中位数；插值也放在去重之后，保证时间轴严格保持 1 天等间隔。  

survey_score 的"有信息缺失"​  

这 18 个缺失（5.00%）来自问卷拒答，而且不是随机漏填：分数低的受访者更不愿填。所以既不删行、也不用均值或中位数填补。删行等于把低分样本系统性剔掉，均值、回归系数、综合得分会偏高；填补则会抹掉"拒答"本身携带的信息。做法是保留 NaN，另加一列 0/1 指示变量 survey_score_missing（1 = 拒答），让"有没有作答"也能进模型。不涉及这个变量的分析，仍可用全部 360 条数据。  

为什么不能删行  

日度面板本来就不宜删行：删掉 18 行会破坏 1 天等间隔（出现 2 天跳空），各区域样本占比最多偏移 0.73 个百分点，插值、时间序列回归、聚类都会受影响。只有确实要拿 survey_score 当输入的模型，才在那一步内部用完整样本（n=342），并做敏感性分析：假设①拒答者取观测分数的 25 分位数、②取所在区域均值，两种极端情况各跑一遍。〔填：排序/系数/结论是否不变〕，说明〔填：结论的稳健程度〕。  

| 变量 | 缺失率 | 成因 | 处理方式 | 理由 |
| --- | --- | --- | --- | --- |
| `household_income` | 88.61% | 几乎未采集 | 删列 | 保住 n = 360 |
| `pm25` | 4.44% | 特殊符号、未记录 | 按区域中位数填补 | 尊重区域差异 |
| `temperature` | 1.11% | 设备故障连续断测 | 线性插值 | 保持 1 天等间隔 |
| `traffic_index` | 2.50% | 未记录 | 中位数填补 | 抗异常值 |
| `fuel_type` | 2.22% | 问卷空填 | 众数填补 | 分类变量 |
| `survey_score` | 5.00% | 问卷拒答，非随机缺失 | 保留 NaN + 0/1 指示变量 | 缺失本身携带信息 |

假设与局限：我们把 survey_score 的拒答当作"带信息的缺失"而非可忽略的缺失；凡涉及该变量的结论，都以上述敏感性分析为前提。  

类型错误时常用  
astype()  
to_numeric()转为数值型  
to_datatime()转为时间序列  
建模前检查常用  
df.info()信息  
df.describe()描述性统计  
df.isna().sum()计算缺失值总数  
df.duplicated().sum()重复行总数  

异常值：数据错误或真实存在极端情况  
1、录入错误：多输数字、单位填错、符号写反  
2、真实极端样本：超大城市、极端天气、特殊年份  
3、模型敏感点：数值本身没错，但是显著影响均值、回归线或聚类结果，一般用到的少  
异常值不是一发现就删除，先判断为什么异常，是录入还是真实值异常（核心）  
录入异常：通勤1200分钟，GDP多个0，通常需要修正和删除  
真实极端样本：虽然极端，但可能是真实现象，需要解释，例如上海重庆交通复杂，通勤时间长正常  

用3σ原则判断数据是否异常，所有数据应在μ±3σ之间  
1、计算μ均值σ标准差，注意标准差根号里面分母为n-1  
2、得到区域  
3、判断数据点是否位于±3σ区域  

In [5]:
# -*- coding: utf-8 -*-
# 3σ 异常值检测（检测列 / 分组列 / 输出名来自 CFG）
# 单独运行本格前，请先运行第 1 个代码单元
def flag_3sigma(s, iters=3):
    """迭代 3σ；mask 单调累加，避免阈值变化把旧异常点撤销。"""
    m = pd.Series(False, index=s.index)
    for _ in range(iters):
        cur = s[s.notna() & ~m]
        mu, sd = cur.mean(), cur.std(ddof=1)
        if pd.isna(sd) or sd == 0:
            break
        new = s.notna() & ((s - mu).abs() > 3 * sd)
        merged = m | new
        if merged.equals(m):
            break
        m = merged
    return m


d = load_any(OUT_DIR, CFG["clean_stem"])
d = d.drop(columns=[c for c in d.columns if c.startswith("flag_") or c.endswith("_out")])
NUM = [c for c in CFG["detect_cols"] if c in d.columns]
GROUP = CFG["group_col"] if CFG["group_col"] in d.columns else None
for c in NUM:
    d[c + "_out"] = (d.groupby(GROUP)[c].transform(lambda s: flag_3sigma(s).astype(int))
                     if GROUP else flag_3sigma(d[c]).astype(int))
d["flag_3sigma"] = d[[c + "_out" for c in NUM]].max(axis=1)
print("3σ 各列:", {c: int(d[c + "_out"].sum()) for c in NUM},
      "| 异常行:", int(d.flag_3sigma.sum()))
save_both(d, OUT_DIR, CFG["out_3sigma"], "3sigma")

3σ 各列: {'pm25': 2, 'temperature': 0, 'traffic_index': 4, 'survey_score': 0} | 异常行: 6


IQR方法  
箱线图把Q1,中位数，Q3上下界和异常点全部展示出来  
IQR=Q_3-Q_1  
Q_1为第一四分位数，Q_3为第三四分位数  
IQR表示中间50%数据的跨度  
认为所有数据应该属于[Q_1-1.5IQR,Q_3+1.5IQR]  
在区间之外视为异常值  
由于IQR依赖四分位数，而不是均值标准差，因此不容易被极端值本身拉偏  
数据比较少常用IQR  

In [6]:
# IQR 异常值检测（检测列 / 分组列 / 输出名来自 CFG）
# 单独运行本格前，请先运行第 1 个代码单元
def flag_iqr(s):
    """1.5×IQR；按两次取分位数，避免解包顺序隐患。"""
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return s.notna() & ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr))


d = load_any(OUT_DIR, CFG["clean_stem"])
d = d.drop(columns=[c for c in d.columns if c.startswith("flag_") or c.endswith("_out")])
NUM = [c for c in CFG["detect_cols"] if c in d.columns]
GROUP = CFG["group_col"] if CFG["group_col"] in d.columns else None
for c in NUM:
    d[c + "_out"] = (d.groupby(GROUP)[c].transform(lambda s: flag_iqr(s).astype(int))
                     if GROUP else flag_iqr(d[c]).astype(int))
d["flag_iqr"] = d[[c + "_out" for c in NUM]].max(axis=1)
print("IQR 各列:", {c: int(d[c + "_out"].sum()) for c in NUM},
      "| 异常行:", int(d.flag_iqr.sum()))
save_both(d, OUT_DIR, CFG["out_iqr"], "IQR")

IQR 各列: {'pm25': 5, 'temperature': 0, 'traffic_index': 8, 'survey_score': 3} | 异常行: 16


In [7]:
# -*- coding: utf-8 -*-
# 按分组列画箱线图：橙圈 = 3σ / IQR 标记的异常点（全部来自 CFG）
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
COLORS = ["#8ecae6", "#a7d7a5", "#ffd28a", "#f4a6a6"]


def draw(method):
    stem = CFG["out_iqr"] if method == "IQR" else CFG["out_3sigma"]
    d = load_any(OUT_DIR, stem)
    NUM = [c for c in CFG["detect_cols"] if c in d.columns][:4]   # 2×2 子图最多 4 列
    # 新数据没有分组列时退化成「全体」一组，保证照样能画
    G = CFG["group_col"] if CFG["group_col"] in d.columns else None
    regions = sorted(d[G].dropna().unique()) if G else ["全体"]
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), dpi=120)
    for ax, c in zip(axes.flat, NUM):
        groups = ([d.loc[d[G] == r, c].dropna().values for r in regions] if G
                  else [d[c].dropna().values])
        bp = ax.boxplot(groups, positions=range(1, len(regions) + 1), widths=0.5,
                        patch_artist=True, showfliers=False, showmeans=True,
                        medianprops={"color": "#c0392b", "linewidth": 2},
                        meanprops={"marker": "D", "markerfacecolor": "#2c3e50",
                                   "markeredgecolor": "#2c3e50", "markersize": 5})
        for box, color in zip(bp["boxes"], COLORS):
            box.set(facecolor=color, edgecolor="#333333", alpha=0.9)
        for k in ("whiskers", "caps"):
            for line in bp[k]:
                line.set(color="#555555", linewidth=1.2)
        for i, r in enumerate(regions, start=1):
            sub = (d[(d[G] == r) & (d[c + "_out"] == 1)] if G
                   else d[d[c + "_out"] == 1])
            if len(sub):
                ax.scatter([i] * len(sub), sub[c], s=45, facecolors="none",
                           edgecolors="#e67e22", linewidths=1.5, zorder=3)
        ax.set_xticks(range(1, len(regions) + 1), regions)
        ax.set_title(c, fontsize=11)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.set_axisbelow(True)
    fig.suptitle(f"按{CFG['group_label']}分组箱线图｜橙圈 = {method} 标记的异常"
                 f"（箱体 IQR、红线中位数、菱形均值）", fontsize=13)
    fig.tight_layout()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out = OUT_DIR / f"{CFG['plot_prefix']}_{method}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("已输出：", out)


draw("IQR")
draw("3sigma")

已输出： d:\git项目\输出数据\boxplot_IQR.png
已输出： d:\git项目\输出数据\boxplot_3sigma.png


先看中位数，再看箱体宽度，最后看是否存在离群点  
箱体越宽说明波动越大；异常点需要结合业务背景判断  
用途：发现异常值，比较不同组数据分布差异  

异常值处理方式：  
1、删除：明确录入错误，删除后不影响样本代表性  
2、替换：有可靠修正值，或用分组统计量填补  
3、缩尾：把超过上下界值压缩到边界，降低极端值影响  
例xi*=min(xi,Q3+1.5IQR)表示压缩超过的异常值为上界  
4、保留并说明：异常值为真实样本，可保留并在论文中说明  

In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path
import pandas as pd

OUT_DIR = Path(r"d:\git项目\输出数据")
NUM = ["pm25", "temperature", "traffic_index", "survey_score"]
GROUP = "region"
STEM = "result_iqr"          # 换成 "result_3sigma" 可切换判定来源
POLICY = "keep"              # drop / replace / winsorize / keep

def load_any(folder, stem):
    for ext in (".csv", ".xlsx"):
        p = folder / f"{stem}{ext}"
        if p.exists():
            return pd.read_excel(p) if ext == ".xlsx" else pd.read_csv(p)
    raise FileNotFoundError(stem)

raw = load_any(OUT_DIR, STEM)
flag = "flag_iqr" if STEM == "result_iqr" else "flag_3sigma"

# 按 region 分组的 1.5×IQR 上下界与分组中位数
bounds, meds = {}, {}
for c in NUM:
    g = raw.groupby(GROUP)[c]
    q1 = g.transform(lambda s: s.quantile(.25))
    q3 = g.transform(lambda s: s.quantile(.75))
    iqr = q3 - q1
    bounds[c] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    meds[c] = g.transform("median")

def apply_policy(policy):
    d = raw.copy()
    if policy == "drop":                        # ① 删除：明确录入错误且不影响代表性
        d = d[d[flag] == 0].copy()
    elif policy in ("replace", "winsorize"):
        for c in NUM:
            bad = d[c + "_out"] == 1
            lo, hi = bounds[c]
            if policy == "replace":             # ② 替换：分组中位数
                d[c] = d[c].mask(bad, meds[c])
            else:                               # ③ 缩尾：压到 IQR 边界
                d[c] = d[c].clip(lower=lo, upper=hi)
    # ④ keep：数值原样，靠标记列 + 论文说明
    d["outlier_policy"] = policy
    return d

# 四种策略对比（敏感性分析，只打印不落盘）
rows = []
for p in ["keep", "winsorize", "replace", "drop"]:
    d = apply_policy(p)
    rows.append({"策略": p, "行数": len(d),
                 "pm25均值": round(d["pm25"].mean(), 2),
                 "pm25标准差": round(d["pm25"].std(), 2),
                 "traffic均值": round(d["traffic_index"].mean(), 2)})
print(pd.DataFrame(rows).to_string(index=False))

out = apply_policy(POLICY)                      # 只输出选定策略的唯一一份结果
out.to_csv(OUT_DIR / "final_model_input.csv", index=False, encoding="utf-8-sig")
out.to_excel(OUT_DIR / "final_model_input.xlsx", index=False, sheet_name="建模输入")
print("已输出：", OUT_DIR / "final_model_input.xlsx", "| 策略：", POLICY)


指标同向化与正向化  
量纲不同，例如单位万元，分钟，无单位  
尺度不同，例如GDP几万，满意度0-100，比例0-1  
方向不同，有些指标越大越好，有些越小越好  

目标是把不同种类数据转化成可以共同进入模型的数据  

指标方向  
效益型：越大越好  
成本型：越小越好  
中间型：接近目标值好  
区间型：落在合理区间最好  

评价类模型与同向化处理，极差标准化挂钩  
回归、机器学习用zscore标准化  

min-max标准化  
将数据压缩到0-1，常用于综合评价和TOPSIS  
综合打分更直观  

In [ ]:
import numpy as np
import pandas as pd

# 换题目只改这里：列 -> (类型, 参数)
KINDS = {
    "pm25":          ("cost", None),           # 成本型：越小越好
    "temperature":   ("intermediate", 22),    # 中间型：22℃ 最优
    "traffic_index": ("cost", None),
    "survey_score":  ("benefit", None),        # 效益型：越大越好
    # "某指标":       ("interval", (40, 60)), # 区间型：[a,b] 内最优
}

def positivize(X, kinds=KINDS):
    """四类指标统一转成「越大越好」并落到 [0,1]；缺失值保留 NaN。"""
    out = pd.DataFrame(index=X.index)
    for col, (kind, p) in kinds.items():
        s = X[col].astype(float)
        rng = s.max() - s.min()
        rng = rng if pd.notna(rng) and rng != 0 else 1
        if kind == "benefit":
            y = (s - s.min()) / rng                      # (x-min)/(max-min)
        elif kind == "cost":
            y = (s.max() - s) / rng                      # (max-x)/(max-min)
        elif kind == "intermediate":
            d = (s - p).abs()
            y = 1 - d / (d.max() if d.max() > 0 else 1)  # 1-|x-x*|/max|x-x*|
        elif kind == "interval":
            a, b = p
            M = max(a - s.min(), s.max() - b)
            M = M if pd.notna(M) and M != 0 else 1
            y = pd.Series(np.nan, index=s.index, dtype=float)
            y[s < a] = 1 - (a - s[s < a]) / M
            y[s > b] = 1 - (s[s > b] - b) / M
            y[s.between(a, b)] = 1.0
        else:
            raise ValueError(f"未知类型：{kind}")
        out[col] = y
    return out

df = pd.read_csv(r"D:\git项目\输出数据\final_cleaned.csv", encoding="utf-8-sig")
X = positivize(df)                                             # ① 分类同向化
X = (X - X.min()) / (X.max() - X.min()).replace(0, 1)         # ② 统一 min-max 到 [0,1]
print(X.describe().round(3))
X.to_csv(r"D:\git项目\输出数据\minmax标准化.csv", index=False, encoding="utf-8-sig")


z-score标准化  
z=(x-μ)/σ  
x本身近似正态，z近似标准正态  
常用于回归、聚类和机器学习  
模型依赖距离、梯度或统计尺度  

In [ ]:
import pandas as pd

COLS = ["pm25", "temperature", "traffic_index", "survey_score"]

def zfit(X):
    """只在训练集上统计；std 用 ddof=1（分母 n-1），常数列给 1 防除零。"""
    return X.mean(), X.std(ddof=1).replace(0, 1)

def ztransform(X, mean, std):
    return (X - mean) / std

df = pd.read_csv(r"D:\git项目\输出数据\final_cleaned.csv", encoding="utf-8-sig")
mean, std = zfit(df[COLS])
X_z = ztransform(df[COLS], mean, std)
print(X_z.describe().round(3))     # mean≈0、std≈1；NaN 保留

# 新数据用同一套参数，不会数据泄漏：
# X_new = ztransform(new_df[COLS], mean, std)


可视化、相关性与假设检验  
可视化：看趋势，时间序列是否上升、下降或周期波动  
看差异，不同对象哪个高低  
看关系，两个变量是否一起变化  
看异常，是否存在明显偏离主体分布的点  
图表不是装饰，每一张图都要服务于问题分析和结果解释  

图表选择  
折线图——时间趋势；销量；温度；交通流  
柱状图——比较对象大小：城市得分、指标排名  
散点图——看两个变量关系：GDP与满意度  
箱线图——看分布、四分位数和异常值  
热力图——看相关矩阵或二维强弱关系  
直方图——看单个变量的分布形状  

描述性统计：先用数字概括数据  
均值、方差、标准差  
在建模中快速判断指标范围、离散程度、异常值和是否需要标准化  

In [ ]:
import pandas as pd

DATA = r"D:/git项目/输出数据/final_cleaned.csv"
COLS = ["pm25", "temperature", "traffic_index", "survey_score"]

def describe(X, cols=COLS):
    """描述性统计：缺失、集中趋势、离散程度、分位数、分布形状；NaN 自动跳过。"""
    s = X[cols]
    t = pd.DataFrame({
        "有效n": s.count(), "缺失": s.isna().sum(),
        "均值": s.mean(), "标准差": s.std(ddof=1),
        "最小值": s.min(), "Q1": s.quantile(.25), "中位数": s.median(),
        "Q3": s.quantile(.75), "最大值": s.max(),
        "偏度": s.skew(), "峰度": s.kurt(),
    })
    t["变异系数CV"] = t["标准差"] / t["均值"]
    return t.round(3)

df = pd.read_csv(DATA, encoding="utf-8-sig")
print(describe(df).to_string())
# 需要保存时去掉下一行注释：
# describe(df).to_csv(r"D:/git项目/输出数据/描述性统计.csv", encoding="utf-8-sig")


相关性分析：变量之间是否一起变化  
一个变量增大另一个也倾向增大时，为正相关  
一个变量减小另一个也倾向减小时，为负相关  
如果没有明显规律，称相关性较弱  

数模中用于判断变量之间是否有关联；  
为回归分析、特征选择、指标筛选提供依据；  
帮助发现多重共线性风险；  
在论文问题分析部分支撑建模思路  
注意：相关性不等于因果性，例如GDP与满意度相关不代表GDP一定导致满意度提高，还需要结合背景和模型进一步分析  

散点图可以给出直观判断，还需要用pearson相关系数  r

假设检验  
先提出一个可以被数据推翻的原假设H0，再用样本判断是否有足够证据拒绝它  
H0通常表示没有差异、没有关联、没有效果  
H1表示有差异、有关联、有效果  
选定显著性水平α，常取0.05或0.01  
根据数据算出检验统计量，再得到p值  
p值是假定H0成立时，出现当前样本或更极端结果的概率  
p值小于α时拒绝H0，认为差异或关联在统计上显著 
此时称有1-α把握证明H0不正确 
p值不是H0为真的概率，也不能证明H1一定成立  
显著不等于效应大，样本量很大时微小差异也会显著  
不拒绝H0不等于接受H0，可能只是样本量不足或检验功效不够  
论文里要同时报样本量、统计量、p值和效应量或置信区间  

In [ ]:
import pandas as pd
from scipy import stats

DATA = r"D:/git项目/输出数据/final_cleaned.csv"
COLS = ["pm25", "temperature", "traffic_index", "survey_score"]
GROUP = "region"

def normality(X, cols):
    """Shapiro-Wilk 正态性：p<0.05 偏离正态，后续考虑非参数检验。"""
    rows = []
    for c in cols:
        r = stats.shapiro(X[c].dropna())
        rows.append(dict(变量=c, W=round(r.statistic, 4), p值=round(r.pvalue, 5),
                         结论="近似正态" if r.pvalue >= .05 else "偏离正态"))
    return pd.DataFrame(rows)

def t_one(X, col, mu0, alternative="two-sided"):
    """单样本 t 检验：该列均值是否等于 mu0。"""
    r = stats.ttest_1samp(X[col].dropna(), mu0, alternative=alternative)
    return dict(检验=f"{col} 均值 vs {mu0}", 统计量=round(r.statistic, 4),
                类型="t", p值=round(r.pvalue, 5), 备注=alternative)

def t_two(X, col, group, a, b):
    """两独立样本 t 检验：先 Levene 判方差齐，再自动选 equal_var。"""
    s1 = X.loc[X[group] == a, col].dropna()
    s2 = X.loc[X[group] == b, col].dropna()
    eq = stats.levene(s1, s2).pvalue >= .05
    r = stats.ttest_ind(s1, s2, equal_var=eq)
    return dict(检验=f"{col}: {a} vs {b}", 统计量=round(r.statistic, 4),
                类型="t", p值=round(r.pvalue, 5), 备注=f"方差齐={eq}")

def chi2(X, c1, c2):
    """卡方独立性检验：两个类别变量是否有关联。"""
    tab = pd.crosstab(X[c1], X[c2])
    r = stats.chi2_contingency(tab)
    return dict(检验=f"{c1} × {c2}", 统计量=round(r.statistic, 4),
                类型="卡方", p值=round(r.pvalue, 5), 备注=f"df={r.dof}")

df = pd.read_csv(DATA, encoding="utf-8-sig")
print(normality(df, COLS).to_string(index=False))       # 前置：决定参数 / 非参数
res = [t_one(df, "pm25", 50, "greater"),
       t_two(df, "pm25", GROUP, "东部", "西部"),
       chi2(df, GROUP, "fuel_type")]
print(pd.DataFrame(res).to_string(index=False))


假设性检验是看有没有线性关系，如果有再看r线性关系强弱
若无线性关系，r再接近1也没用
因此假设性检验先于pearson相关系数

常用检验与适用条件  
单样本t检验：一组样本均值是否等于给定值，要求数据近似正态  
两独立样本t检验：两组均值是否有差异，先做方差齐性检验决定版本  
配对t检验：同一批对象前后两次测量，例如政策实施前后  
卡方检验：两个类别变量是否独立，用列联表，期望频数不宜小于5  
相关系数显著性检验：pearson相关系数是否显著不为0  
正态性检验：Shapiro-Wilk或K-S，决定能不能用参数方法  
方差齐性检验：Levene，决定t检验用哪一个版本  
非参数替代：明显偏态或样本很小时用Mann-Whitney U、Kruskal-Wallis  
多重检验：同时做很多次检验要做Bonferroni或FDR校正，否则假阳性会变多  

相关性检验和t检验适合放在“数据分析”、“变量筛选”、“模型检验”、“结果分析”四个部分

相关性分析与假设检验适合在  
“问题分析”说明变量之间可能存在关系，为模型选择提供依据  
“数据预处理”说明异常值，分布特征和变量关系的检查过程  
“模型检验”说明模型结果是否稳定，变量关系是否显著 

例子1 某地区pm25均值是否超过标准50  
H0为均值不超过50，H1为超过50，用单样本右侧t检验  
若p=0.012，在α=0.05下拒绝H0，认为该地区显著超标  
论文里要同时给样本量、均值、置信区间和p值，不能只报p值  

例子2 东部与西部满意度是否有差异  
H0为两地均值相等，用两独立样本t检验  
若p=0.31，不能拒绝H0，应写没有足够证据认为两地不同，而不是两地相同   

例子3 散点图显示pm25与满意度负相关，是否可信  
对pearson相关系数做显著性检验，H0为总体相关系数为0  
样本量很大时很小的r也会显著，所以必须同时报告r的大小  
相关不等于因果，两者可能同时受温度、交通、收入影响  

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

DATA = r"D:/git项目/输出数据/final_cleaned.csv"
COLS = ["pm25", "temperature", "traffic_index", "survey_score"]

def corr_test(X, cols, method="pearson", alpha=.05):
    """逐对算 r 及其显著性 t 检验：t = r*sqrt((n-2)/(1-r^2)), df = n-2。"""
    rows = []
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            pair = X[[a, b]].dropna()          # 成对删除，保留最多可用样本
            n = len(pair)
            r = pair[a].corr(pair[b], method=method)
            t = r * np.sqrt((n - 2) / (1 - r * r)) if abs(r) < 1 else np.inf
            p = 2 * stats.t.sf(abs(t), df=n - 2)
            rows.append(dict(变量1=a, 变量2=b, n=n, r=round(r, 4),
                             t=round(t, 3), p值=round(p, 5),
                             结论="显著相关" if p < alpha else "不显著"))
    return pd.DataFrame(rows)

df = pd.read_csv(DATA, encoding="utf-8-sig")
print(df[COLS].corr().round(3).to_string())       # 相关系数矩阵
print(corr_test(df, COLS).to_string(index=False))
# 非参数版本：corr_test(df, COLS, method="spearman")
